# 03 — Clean & Harmonise Data

Raw data from OECD DAC and OCHA FTS uses different vocabularies, currencies, and donor
name conventions. Before we can compare or combine them, we need to:

| Step | Problem | Fix |
|------|---------|-----|
| 1 | OECD uses `DonorName`; FTS uses `source_org` — different strings for the same country | Build a canonical donor name table |
| 2 | OECD CRS purpose codes are 5-digit numbers; FTS uses cluster names | Map both to a shared sector taxonomy |
| 3 | All amounts are nominal USD — a 2018 dollar ≠ a 2024 dollar | Deflate to constant 2023 USD using World Bank CPI |
| 4 | Neither dataset has a consistent region grouping | Add UN geoscheme regions for aggregation |
| 5 | Both datasets have quality issues (zeros, nulls, duplicates) | Filter and flag |

**Outputs:**
- `data/processed/oecd_clean.csv` — OECD DAC flows, cleaned and deflated
- `data/processed/fts_clean.csv` — FTS flows, cleaned and deflated
- `data/processed/combined.csv` — Both sources stacked with a `source` column

These are the inputs for `04_gap_analysis.ipynb`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

OECD_RAW   = PROJECT_ROOT / "data" / "raw" / "oecd_dac_2018_2024.csv"
FTS_RAW    = PROJECT_ROOT / "data" / "raw" / "fts_flows_2018_2025.csv"
PROC_DIR   = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

print(f"OECD raw: {OECD_RAW}  (exists: {OECD_RAW.exists()})")
print(f"FTS raw:  {FTS_RAW}   (exists: {FTS_RAW.exists()})")

## Step 1 — Load raw data

Load only the columns we need from each dataset. The OECD file is 168 MB with ~2M rows;
loading everything would use ~4 GB of RAM. We select columns up front.

In [ ]:
# ── OECD DAC — column selection ───────────────────────────────────────────────
# The CRS dataset has ~80 columns. We need only a subset for gap analysis.
# See the OECD CRS codebook for full column definitions:
# https://www.oecd.org/dac/financing-sustainable-development/development-finance-standards/crs-overview.htm

OECD_COLS = {
    # Donor
    "DonorName": "donor_name",
    "DonorCode": "donor_code",

    # Recipient
    "RecipientName": "recipient_name",
    "RecipientCode": "recipient_code",
    "RegionName": "region_name",

    # Amount
    "USD_Disbursement": "disbursement_usd",
    "USD_Commitment": "commitment_usd",

    # Classification
    "Year": "year",
    "PurposeName": "purpose_name",
    "PurposeCode": "purpose_code",
    "SectorName": "sector_name",
    "SectorCode": "sector_code",
    "FlowName": "flow_type",       # ODA grant, ODA loan, etc.
    "ChannelName": "channel_name", # bilateral, multilateral, NGO, etc.
    "AidTypeName": "aid_type",
}

print("Loading OECD data (selecting columns)...")
oecd_raw = pd.read_csv(
    OECD_RAW,
    usecols=lambda c: c in OECD_COLS,
    low_memory=False,
)
oecd_raw = oecd_raw.rename(columns=OECD_COLS)

print(f"  Loaded: {len(oecd_raw):,} rows × {oecd_raw.shape[1]} columns")
print(f"  Years:  {sorted(oecd_raw['year'].unique())}")
print(f"  Donors: {oecd_raw['donor_name'].nunique()}")
print(f"\nColumn dtypes:\n{oecd_raw.dtypes}")

In [ ]:
# ── FTS — already lean, load everything ───────────────────────────────────────
print("Loading FTS data...")
fts_raw = pd.read_csv(FTS_RAW, dtype={"flow_id": str}, low_memory=False)
fts_raw["amount_usd"] = pd.to_numeric(fts_raw["amount_usd"], errors="coerce").fillna(0)
fts_raw["date"] = pd.to_datetime(fts_raw["date"], errors="coerce")
fts_raw["year"] = fts_raw["query_year"].astype(int)

print(f"  Loaded: {len(fts_raw):,} rows × {fts_raw.shape[1]} columns")
print(f"  Years:  {sorted(fts_raw['year'].unique())}")
print(f"  Source orgs: {fts_raw['source_org'].nunique()}")

## Step 2 — Standardise donor names

The same country appears under different strings in OECD and FTS:

| Country | OECD DonorName | FTS source_org |
|---------|---------------|----------------|
| USA | `United States` | `United States of America, Government of` |
| Germany | `Germany` | `Germany, Government of` |
| UK | `United Kingdom` | `United Kingdom of Great Britain and Northern Ireland, Government of` |

We build a lookup table mapping raw strings → canonical short names (ISO-style).
This lets us join and compare the two datasets in gap analysis.

**Approach:** We keep it simple — a hand-curated dictionary for the top 30 donors
that account for ~95% of all funding. Everything else maps to its raw name.

In [ ]:
# Inspect what the top donor names look like in each dataset.
# This tells us exactly what strings we need to map.

print("Top 20 donor names in OECD (by disbursement):")
top_oecd_donors = (
    oecd_raw.groupby("donor_name")["disbursement_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
for name, val in top_oecd_donors.items():
    print(f"  {val/1e9:6.1f}B  {name!r}")

print("\nTop 20 donor names in FTS (by amount):")
top_fts_donors = (
    fts_raw.groupby("source_org")["amount_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)
for name, val in top_fts_donors.items():
    print(f"  {val/1e9:6.1f}B  {name!r}")

In [ ]:
# ── Canonical donor name lookup ───────────────────────────────────────────────
# Keys: raw strings from OECD or FTS. Values: canonical short name.
# We cover the top donors that appear in both datasets; the rest pass through unchanged.

DONOR_MAP = {
    # United States
    "United States": "United States",
    "United States of America, Government of": "United States",
    "USAID": "United States",
    "United States Department of State": "United States",
    "United States Department of Defense": "United States",

    # European Union
    "EU Institutions": "European Union",
    "European Commission": "European Union",
    "European Commission - Directorate-General for European Civil Protection and Humanitarian Aid Operations": "European Union",
    "European Commission - DG ECHO": "European Union",

    # Germany
    "Germany": "Germany",
    "Germany, Government of": "Germany",
    "Federal Republic of Germany": "Germany",

    # United Kingdom
    "United Kingdom": "United Kingdom",
    "United Kingdom of Great Britain and Northern Ireland, Government of": "United Kingdom",
    "UK - Foreign, Commonwealth & Development Office": "United Kingdom",

    # Japan
    "Japan": "Japan",
    "Japan, Government of": "Japan",

    # France
    "France": "France",
    "France, Government of": "France",

    # Sweden
    "Sweden": "Sweden",
    "Sweden, Government of": "Sweden",

    # Norway
    "Norway": "Norway",
    "Norway, Government of": "Norway",

    # Canada
    "Canada": "Canada",
    "Canada, Government of": "Canada",

    # Netherlands
    "Netherlands": "Netherlands",
    "Netherlands, Government of": "Netherlands",

    # Denmark
    "Denmark": "Denmark",
    "Denmark, Government of": "Denmark",

    # Switzerland
    "Switzerland": "Switzerland",
    "Switzerland, Government of": "Switzerland",

    # Australia
    "Australia": "Australia",
    "Australia, Government of": "Australia",

    # South Korea
    "Korea": "South Korea",
    "Republic of Korea, Government of": "South Korea",

    # Saudi Arabia
    "Saudi Arabia": "Saudi Arabia",
    "Saudi Arabia, Government of": "Saudi Arabia",

    # UAE
    "United Arab Emirates": "United Arab Emirates",
    "United Arab Emirates, Government of": "United Arab Emirates",

    # Turkey
    "Türkiye": "Turkey",
    "Turkey": "Turkey",
    "Republic of Türkiye, Government of": "Turkey",
    "Republic of Turkey, Government of": "Turkey",

    # Kuwait
    "Kuwait": "Kuwait",
    "Kuwait, Government of": "Kuwait",
}

def standardise_donor(raw_name: str) -> str:
    """Map a raw donor string to a canonical short name."""
    if pd.isna(raw_name):
        return "Unknown"
    return DONOR_MAP.get(raw_name, raw_name)

# Apply to both datasets
oecd_raw["donor_canonical"] = oecd_raw["donor_name"].map(standardise_donor)
fts_raw["donor_canonical"] = fts_raw["source_org"].map(standardise_donor)

# Spot-check: verify US looks right
print("OECD — US canonical check:")
print(oecd_raw[oecd_raw["donor_canonical"] == "United States"][["donor_name", "donor_canonical"]].drop_duplicates())

print("\nFTS — US canonical check:")
print(fts_raw[fts_raw["donor_canonical"] == "United States"][["source_org", "donor_canonical"]].drop_duplicates())

## Step 3 — Map sectors to a shared taxonomy

**OECD CRS** uses a 5-digit purpose code hierarchy. The first 3 digits are the sector
(e.g. `720` = Emergency Response). The last 2 specify the purpose (e.g. `72010` =
Material relief assistance and services). We collapse these into ~10 readable categories.

**FTS** uses `GlobalCluster` names (Food Security, Health, Shelter, etc.) — a
UN-standard humanitarian taxonomy. We map these to the same readable categories.

The shared taxonomy is deliberately coarse — fine-grained sector comparison between
the two datasets isn't possible because they use fundamentally different classifications.

In [ ]:
# First: inspect what OECD sector codes exist in the data
print("OECD sector codes present (top 20 by disbursement):")
print(
    oecd_raw.groupby(["sector_code", "sector_name"])["disbursement_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)

In [ ]:
# ── OECD sector code → shared taxonomy ───────────────────────────────────────
# OECD 3-digit sector codes: https://www.oecd.org/dac/financing-sustainable-development/
# development-finance-standards/dacandcrscodelists.htm

OECD_SECTOR_MAP = {
    # Emergency / Humanitarian
    720: "Emergency & Humanitarian",    # Emergency Response
    730: "Emergency & Humanitarian",    # Reconstruction Relief
    740: "Emergency & Humanitarian",    # Disaster Prevention

    # Health
    120: "Health",                      # Health, General
    121: "Health",                      # Health, Basic
    122: "Health",                      # Basic Health
    130: "Health",                      # Population Policies

    # Food & Agriculture
    311: "Food & Agriculture",          # Agriculture
    312: "Food & Agriculture",          # Forestry
    313: "Food & Agriculture",          # Fishing
    321: "Food & Agriculture",          # Industry
    520: "Food & Agriculture",          # Developmental food assistance

    # Education
    110: "Education",
    111: "Education",
    112: "Education",
    113: "Education",

    # Water & Sanitation
    140: "Water & Sanitation",

    # Governance & Civil Society
    151: "Governance & Civil Society",
    152: "Governance & Civil Society",
    160: "Governance & Civil Society",  # Other Social Infrastructure

    # Refugee Support
    930: "Refugees & Migration",        # Refugees in Donor Countries

    # Debt
    600: "Debt Relief",

    # Multisector
    400: "Multisector / Environment",
    410: "Multisector / Environment",
    430: "Multisector / Environment",
}

def map_oecd_sector(code) -> str:
    """Map a 5-digit OECD purpose code to a shared sector label."""
    if pd.isna(code):
        return "Unspecified"
    # Take the first 3 digits to get the sector group
    sector_3 = int(str(int(code))[:3])
    return OECD_SECTOR_MAP.get(sector_3, "Other")

oecd_raw["sector_clean"] = oecd_raw["purpose_code"].apply(map_oecd_sector)

print("Sector distribution in OECD data ($ billion):")
print(
    oecd_raw.groupby("sector_clean")["disbursement_usd"]
    .sum()
    .sort_values(ascending=False)
    .div(1e9)
    .round(2)
    .to_string()
)

In [ ]:
# ── FTS GlobalCluster → shared taxonomy ──────────────────────────────────────
# FTS cluster names are already readable; we just map them to the same labels.

FTS_CLUSTER_MAP = {
    "Food Security": "Food & Agriculture",
    "Nutrition": "Health",
    "Health": "Health",
    "Water Sanitation Hygiene": "Water & Sanitation",
    "WASH": "Water & Sanitation",
    "Shelter and Non-Food Items": "Emergency & Humanitarian",
    "Shelter": "Emergency & Humanitarian",
    "Non-food Items": "Emergency & Humanitarian",
    "Emergency Shelter and NFI": "Emergency & Humanitarian",
    "Protection": "Governance & Civil Society",
    "Education": "Education",
    "Camp Coordination and Camp Management": "Emergency & Humanitarian",
    "CCCM": "Emergency & Humanitarian",
    "Emergency Telecommunications": "Emergency & Humanitarian",
    "Logistics": "Emergency & Humanitarian",
    "Coordination and Support Services": "Emergency & Humanitarian",
    "Multi-Cluster/Sector": "Multisector / Environment",
    "Mine Action": "Governance & Civil Society",
    "Agriculture": "Food & Agriculture",
    "Livelihoods": "Food & Agriculture",
    "Early Recovery": "Emergency & Humanitarian",
}

def map_fts_cluster(cluster) -> str:
    if pd.isna(cluster):
        return "Unspecified"
    return FTS_CLUSTER_MAP.get(cluster, cluster)

fts_raw["sector_clean"] = fts_raw["cluster"].apply(map_fts_cluster)

print("Sector distribution in FTS data ($ billion):")
print(
    fts_raw.groupby("sector_clean")["amount_usd"]
    .sum()
    .sort_values(ascending=False)
    .div(1e9)
    .round(2)
    .to_string()
)

## Step 4 — Convert to constant 2023 USD

All amounts in both datasets are **nominal USD** — the dollar value at the time of the
transaction. To make year-on-year comparisons fair, we deflate to **constant 2023 USD**
using the US GDP deflator (a standard choice for aid flows denominated in USD).

Source: World Bank — NY.GDP.DEFL.ZS (GDP deflator, base year varies by country;
we use the US series, which is already normalised to index = 100 in the base year).

**Formula:**  
`constant_2023_usd = nominal_usd × (deflator_2023 / deflator_year)`

This makes 2018 dollars larger (inflation means a 2018 dollar bought more) and leaves
2023 dollars unchanged.

**Note on World Bank data retrieval:** We use the `wbdata` library (already in
`requirements.txt`). If the World Bank API is unavailable, we fall back to hard-coded
deflator values from the IMF World Economic Outlook (April 2024 vintage).

In [ ]:
import wbdata
import datetime

# ── Fetch US GDP deflator from World Bank ─────────────────────────────────────
# Indicator NY.GDP.DEFL.ZS: GDP deflator (base year varies)
# We re-index to 2023 = 1.0 so the conversion formula is simple.

# IMF WEO April 2024 fallback values (US GDP deflator index, 2023 = 100)
IMF_DEFLATOR = {
    2018: 89.29,
    2019: 90.89,
    2020: 91.98,
    2021: 96.17,
    2022: 104.40,
    2023: 100.00,
    2024: 102.30,
    2025: 104.50,
}

try:
    print("Fetching US GDP deflator from World Bank API...")
    date_range = (datetime.datetime(2018, 1, 1), datetime.datetime(2025, 1, 1))
    wb_data = wbdata.get_dataframe(
        {"NY.GDP.DEFL.ZS": "deflator"},
        country="USA",
        date=date_range,
    )
    deflators_raw = wb_data["deflator"].dropna().to_dict()

    # wbdata returns datetime keys; convert to year integers
    deflators_raw = {
        (k.year if hasattr(k, "year") else int(str(k)[:4])): v
        for k, v in deflators_raw.items()
    }

    # Re-index so 2023 = 100
    base = deflators_raw.get(2023, 100)
    DEFLATOR = {yr: val / base * 100 for yr, val in deflators_raw.items()}
    print(f"  World Bank deflators (2023 = 100): {dict(sorted(DEFLATOR.items()))}")
    source = "World Bank NY.GDP.DEFL.ZS"

except Exception as e:
    print(f"  World Bank API failed ({e}). Using IMF WEO April 2024 fallback.")
    DEFLATOR = IMF_DEFLATOR
    source = "IMF WEO April 2024 (fallback)"

print(f"\nDeflator source: {source}")
print(f"Year | Deflator index (2023=100) | Conversion factor to 2023")
print("-" * 58)
for yr in sorted(DEFLATOR):
    factor = 100 / DEFLATOR[yr]
    print(f"{yr}  |  {DEFLATOR[yr]:20.2f}  |  {factor:.4f}")

In [ ]:
def deflate_to_2023(amount: float, year: int) -> float:
    """Convert a nominal USD amount in `year` to constant 2023 USD."""
    factor = 100 / DEFLATOR.get(year, 100)  # default to 1.0 if year missing
    return amount * factor

# Apply to OECD
oecd_raw["disbursement_2023usd"] = oecd_raw.apply(
    lambda r: deflate_to_2023(r["disbursement_usd"] or 0, r["year"]), axis=1
)
oecd_raw["commitment_2023usd"] = oecd_raw.apply(
    lambda r: deflate_to_2023(r["commitment_usd"] or 0, r["year"]), axis=1
)

# Apply to FTS
fts_raw["amount_2023usd"] = fts_raw.apply(
    lambda r: deflate_to_2023(r["amount_usd"], r["year"]), axis=1
)

# Spot-check: show 2018 vs 2023 values for a large US OECD disbursement
sample = oecd_raw[
    (oecd_raw["donor_canonical"] == "United States") &
    (oecd_raw["year"] == 2018)
].nlargest(3, "disbursement_usd")[["year", "disbursement_usd", "disbursement_2023usd", "purpose_name"]]
print("Sample deflation check (US 2018 disbursements):")
print(sample.to_string())

us_2018_nom = oecd_raw[(oecd_raw["year"]==2018)&(oecd_raw["donor_canonical"]=="United States")]["disbursement_usd"].sum()
us_2018_real = oecd_raw[(oecd_raw["year"]==2018)&(oecd_raw["donor_canonical"]=="United States")]["disbursement_2023usd"].sum()
print(f"\nUS 2018 total disbursement: ${us_2018_nom/1e9:.2f}B nominal → ${us_2018_real/1e9:.2f}B constant 2023")

## Step 5 — Add UN geoscheme regions

OECD has a `region_name` column but it uses DAC regional groupings (e.g. "Africa, South
of Sahara") which don't match FTS country names. FTS has `dest_country` but no region.

We add **UN geoscheme** regions — the standard used by the UN Statistical Division —
to both datasets. This lets us roll up analysis by region ("Sub-Saharan Africa",
"Middle East", etc.) without needing an external join table.

We maintain a compact hand-coded mapping for the ~80 countries that appear in these
datasets. This is faster and more transparent than pulling a full country-region table.

In [ ]:
# UN geoscheme regions for the countries that appear in our datasets.
# Source: https://unstats.un.org/unsd/methodology/m49/
# We use the top-level 5-region grouping: Africa, Americas, Asia, Europe, Oceania.
# Within these we add common sub-regions for humanitarian relevance.

COUNTRY_REGION = {
    # ── Sub-Saharan Africa ──────────────────────────────────────────────────
    "Ethiopia": "Sub-Saharan Africa",
    "Democratic Republic of the Congo": "Sub-Saharan Africa",
    "Congo, Democratic Republic of the": "Sub-Saharan Africa",
    "Sudan": "Sub-Saharan Africa",
    "South Sudan": "Sub-Saharan Africa",
    "Somalia": "Sub-Saharan Africa",
    "Nigeria": "Sub-Saharan Africa",
    "Chad": "Sub-Saharan Africa",
    "Mali": "Sub-Saharan Africa",
    "Niger": "Sub-Saharan Africa",
    "Burkina Faso": "Sub-Saharan Africa",
    "Central African Republic": "Sub-Saharan Africa",
    "Mozambique": "Sub-Saharan Africa",
    "Zimbabwe": "Sub-Saharan Africa",
    "Madagascar": "Sub-Saharan Africa",
    "Malawi": "Sub-Saharan Africa",
    "Zambia": "Sub-Saharan Africa",
    "Kenya": "Sub-Saharan Africa",
    "Uganda": "Sub-Saharan Africa",
    "Tanzania": "Sub-Saharan Africa",
    "Rwanda": "Sub-Saharan Africa",
    "Burundi": "Sub-Saharan Africa",
    "Cameroon": "Sub-Saharan Africa",
    "Liberia": "Sub-Saharan Africa",
    "Sierra Leone": "Sub-Saharan Africa",
    "Guinea": "Sub-Saharan Africa",
    "Senegal": "Sub-Saharan Africa",
    "Angola": "Sub-Saharan Africa",
    "Eswatini": "Sub-Saharan Africa",
    "Lesotho": "Sub-Saharan Africa",

    # ── North Africa ────────────────────────────────────────────────────────
    "Libya": "North Africa & Middle East",
    "Egypt": "North Africa & Middle East",
    "Tunisia": "North Africa & Middle East",
    "Morocco": "North Africa & Middle East",
    "Algeria": "North Africa & Middle East",

    # ── Middle East ─────────────────────────────────────────────────────────
    "Syrian Arab Republic": "North Africa & Middle East",
    "Syria": "North Africa & Middle East",
    "Yemen": "North Africa & Middle East",
    "occupied Palestinian territory": "North Africa & Middle East",
    "Palestinian Territory": "North Africa & Middle East",
    "Gaza Strip": "North Africa & Middle East",
    "West Bank and Gaza": "North Africa & Middle East",
    "Iraq": "North Africa & Middle East",
    "Lebanon": "North Africa & Middle East",
    "Jordan": "North Africa & Middle East",
    "Turkey": "North Africa & Middle East",

    # ── South & Southeast Asia ──────────────────────────────────────────────
    "Afghanistan": "South & Southeast Asia",
    "Pakistan": "South & Southeast Asia",
    "Bangladesh": "South & Southeast Asia",
    "Myanmar": "South & Southeast Asia",
    "Nepal": "South & Southeast Asia",
    "India": "South & Southeast Asia",
    "Philippines": "South & Southeast Asia",
    "Cambodia": "South & Southeast Asia",
    "Indonesia": "South & Southeast Asia",
    "Lao People's Democratic Republic": "South & Southeast Asia",
    "Sri Lanka": "South & Southeast Asia",

    # ── Latin America & Caribbean ────────────────────────────────────────────
    "Haiti": "Latin America & Caribbean",
    "Venezuela": "Latin America & Caribbean",
    "Colombia": "Latin America & Caribbean",
    "Honduras": "Latin America & Caribbean",
    "Guatemala": "Latin America & Caribbean",
    "El Salvador": "Latin America & Caribbean",
    "Nicaragua": "Latin America & Caribbean",
    "Bolivia": "Latin America & Caribbean",
    "Peru": "Latin America & Caribbean",
    "Ecuador": "Latin America & Caribbean",
    "Cuba": "Latin America & Caribbean",

    # ── Eastern Europe / Central Asia ────────────────────────────────────────
    "Ukraine": "Eastern Europe & Central Asia",
    "Georgia": "Eastern Europe & Central Asia",
    "Armenia": "Eastern Europe & Central Asia",
    "Azerbaijan": "Eastern Europe & Central Asia",
    "Kazakhstan": "Eastern Europe & Central Asia",
    "Kyrgyzstan": "Eastern Europe & Central Asia",
    "Tajikistan": "Eastern Europe & Central Asia",
}

def get_region(country: str) -> str:
    if pd.isna(country):
        return "Unspecified / Global"
    return COUNTRY_REGION.get(country, "Other / Unclassified")

# Apply to FTS (uses dest_country)
fts_raw["region"] = fts_raw["dest_country"].map(get_region)

# Apply to OECD (uses recipient_name)
oecd_raw["region"] = oecd_raw["recipient_name"].map(get_region)

print("FTS region coverage:")
print(fts_raw["region"].value_counts().to_string())

## Step 6 — Filter and quality checks

Before writing the cleaned outputs, we apply a set of filters to remove records that
would distort the analysis. We document every filter so readers can reproduce or
challenge our choices.

In [ ]:
print("=== OECD CLEANING FILTERS ===")
n0 = len(oecd_raw)

# Filter 1: Keep only ODA flows (FlowCode 11 was already applied at download time,
# but verify here in case the raw file was re-downloaded without the filter).
# flow_type column has values like 'ODA Grants', 'ODA Loans', etc.
oecd_oda = oecd_raw[oecd_raw["flow_type"].str.startswith("ODA", na=False)].copy()
print(f"Filter 1 — ODA only:            {n0:>8,} → {len(oecd_oda):>8,} rows (-{n0-len(oecd_oda):,})")

# Filter 2: Drop rows where BOTH disbursement and commitment are null/zero.
# These are administrative entries with no financial value.
n1 = len(oecd_oda)
oecd_oda = oecd_oda[
    (oecd_oda["disbursement_usd"].fillna(0) != 0) |
    (oecd_oda["commitment_usd"].fillna(0) != 0)
].copy()
print(f"Filter 2 — non-zero amounts:    {n1:>8,} → {len(oecd_oda):>8,} rows (-{n1-len(oecd_oda):,})")

# Filter 3: Drop rows flagged as 'refugee costs in donor countries'.
# These are domestic spending items counted as ODA — controversial inclusion.
# Sector code 930xx.
n2 = len(oecd_oda)
oecd_oda = oecd_oda[oecd_oda["sector_clean"] != "Refugees & Migration"].copy()
print(f"Filter 3 — excl. in-donor refugee costs: {n2:>8,} → {len(oecd_oda):>8,} rows (-{n2-len(oecd_oda):,})")
print(f"           (in-donor refugee costs = domestic spending classified as ODA)")

n_final_oecd = len(oecd_oda)
print(f"\nFinal OECD rows: {n_final_oecd:,} ({n_final_oecd/n0*100:.1f}% of raw)")

In [ ]:
print("=== FTS CLEANING FILTERS ===")
n0_fts = len(fts_raw)

# Filter 1: Keep only financial flows (exclude in-kind — USD valuation unreliable)
fts_fin = fts_raw[fts_raw["contribution_type"] == "financial"].copy()
print(f"Filter 1 — financial only:      {n0_fts:>8,} → {len(fts_fin):>8,} rows (-{n0_fts-len(fts_fin):,})")

# Filter 2: Remove zero-amount flows (in-kind converted, parked, etc.)
n1_fts = len(fts_fin)
fts_fin = fts_fin[fts_fin["amount_usd"] > 0].copy()
print(f"Filter 2 — non-zero amounts:    {n1_fts:>8,} → {len(fts_fin):>8,} rows (-{n1_fts-len(fts_fin):,})")

# Filter 3: Remove flows where source_org is null (can't attribute to a donor)
n2_fts = len(fts_fin)
fts_fin = fts_fin[fts_fin["source_org"].notna()].copy()
print(f"Filter 3 — known source org:    {n2_fts:>8,} → {len(fts_fin):>8,} rows (-{n2_fts-len(fts_fin):,})")

# Deduplication check: FTS can have duplicate flow IDs when a flow is shared
# across multiple years (behavior='shared'). Our query_year approach means the
# same flow_id may appear in multiple year files.
n3_fts = len(fts_fin)
# Keep only the first appearance of each flow_id to avoid double-counting.
# The first appearance corresponds to the earliest query_year, which is closest
# to when the flow was actually made.
fts_fin = fts_fin.sort_values("year").drop_duplicates(subset="flow_id", keep="first")
print(f"Filter 4 — dedup flow_id:       {n3_fts:>8,} → {len(fts_fin):>8,} rows (-{n3_fts-len(fts_fin):,})")
print(f"           (duplicate IDs arise from multi-year 'shared' flows)")

n_final_fts = len(fts_fin)
print(f"\nFinal FTS rows: {n_final_fts:,} ({n_final_fts/n0_fts*100:.1f}% of raw)")

## Step 7 — Write cleaned outputs

We write three CSVs to `data/processed/`:
- `oecd_clean.csv` — cleaned OECD DAC with deflated amounts and canonical donor names
- `fts_clean.csv` — cleaned FTS with deflated amounts and canonical donor names
- `combined.csv` — both datasets stacked with a `source` tag and a common schema

The combined dataset is the primary input for gap analysis. Its schema is deliberately
minimal — only the columns needed for the gap calculation — to keep it readable.

In [ ]:
# ── Write OECD clean ──────────────────────────────────────────────────────────
OECD_KEEP = [
    "year", "donor_canonical", "donor_name", "donor_code",
    "recipient_name", "recipient_code", "region",
    "disbursement_usd", "disbursement_2023usd",
    "commitment_usd", "commitment_2023usd",
    "sector_clean", "purpose_name", "purpose_code",
    "flow_type", "channel_name", "aid_type",
]
oecd_out = oecd_oda[[c for c in OECD_KEEP if c in oecd_oda.columns]].copy()

out_path = PROC_DIR / "oecd_clean.csv"
oecd_out.to_csv(out_path, index=False)
print(f"OECD clean: {len(oecd_out):,} rows → {out_path} ({out_path.stat().st_size/1e6:.1f} MB)")

# ── Write FTS clean ───────────────────────────────────────────────────────────
FTS_KEEP = [
    "flow_id", "year", "date",
    "donor_canonical", "source_org",
    "dest_org", "dest_country", "region",
    "amount_usd", "amount_2023usd",
    "sector_clean", "cluster",
    "status", "flow_type", "contribution_type",
]
fts_out = fts_fin[[c for c in FTS_KEEP if c in fts_fin.columns]].copy()

out_path_fts = PROC_DIR / "fts_clean.csv"
fts_out.to_csv(out_path_fts, index=False)
print(f"FTS clean:  {len(fts_out):,} rows → {out_path_fts} ({out_path_fts.stat().st_size/1e6:.1f} MB)")

# ── Build combined dataset ────────────────────────────────────────────────────
# Minimal shared schema: source, year, donor, recipient_country, amount_2023usd, sector

oecd_stack = pd.DataFrame({
    "source": "OECD_DAC",
    "year": oecd_out["year"],
    "donor": oecd_out["donor_canonical"],
    "recipient_country": oecd_out["recipient_name"],
    "region": oecd_out["region"],
    "amount_2023usd": oecd_out["disbursement_2023usd"].fillna(0),
    "sector": oecd_out["sector_clean"],
    "channel": oecd_out.get("channel_name", pd.NA),
})

fts_stack = pd.DataFrame({
    "source": "FTS",
    "year": fts_out["year"],
    "donor": fts_out["donor_canonical"],
    "recipient_country": fts_out["dest_country"],
    "region": fts_out["region"],
    "amount_2023usd": fts_out["amount_2023usd"],
    "sector": fts_out["sector_clean"],
    "channel": fts_out.get("dest_org", pd.NA),
})

combined = pd.concat([oecd_stack, fts_stack], ignore_index=True)
combined["is_us"] = combined["donor"] == "United States"

combined_path = PROC_DIR / "combined.csv"
combined.to_csv(combined_path, index=False)
print(f"Combined:   {len(combined):,} rows → {combined_path} ({combined_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
print("=== CLEANING VALIDATION ===")
print()

# 1. US share of OECD ODA (constant 2023 USD, most recent year)
max_yr_oecd = oecd_out["year"].max()
us_oecd = oecd_out[(oecd_out["year"]==max_yr_oecd)&(oecd_out["donor_canonical"]=="United States")]["disbursement_2023usd"].sum()
all_oecd = oecd_out[oecd_out["year"]==max_yr_oecd]["disbursement_2023usd"].sum()
print(f"1. OECD {max_yr_oecd}: US=${us_oecd/1e9:.2f}B / Total=${all_oecd/1e9:.2f}B = {us_oecd/all_oecd*100:.1f}%")

# 2. FTS top donor check
fts_top = fts_out.groupby("donor_canonical")["amount_2023usd"].sum().sort_values(ascending=False).head(5)
print(f"\n2. FTS top 5 donors (all years, constant 2023 USD):")
for donor, val in fts_top.items():
    print(f"   {donor}: ${val/1e9:.2f}B")

# 3. Combined: US vs others by year
print(f"\n3. Combined — US vs others by year (constant 2023 USD $B):")
combined_yr = (
    combined.groupby(["year", "is_us"])["amount_2023usd"]
    .sum()
    .unstack(fill_value=0)
    .rename(columns={True: "US_2023B", False: "Others_2023B"})
    / 1e9
).round(2)
print(combined_yr.to_string())

# 4. Check for any NaN amounts in combined
nan_amounts = combined["amount_2023usd"].isna().sum()
print(f"\n4. NaN amounts in combined: {nan_amounts} (should be 0)")

print("\n✓ Cleaning complete. Outputs in data/processed/")